# H11 · Sensitivity: is the development–ROA link driven by one country or one definition?

> **Hypothesis H11:** The main finding (H0: more developed countries, lower ROA) survives dropping any single country, and does not depend on how we define development or ROA.

| | |
|---|---|
| **Why we ask** | The scatter plot of notebook 03 and the dashboard shows India and the USA far from the other points. With only 34 countries one or two can decide a result. A result that disappears when one country is removed is not a finding about countries. |
| **Prediction** | The coefficient stays **negative and significant (p < 0.10)** in at least 90% of 34 leave-one-country-out runs and of the alternative definitions. |
| **Data** | Yahoo company-years, same model as H0 (`ROA ~ development index + sector + size rank + fiscal year`). Each run drops one country, or changes the definition of development or of ROA. |
| **Primary test (fixed before looking at the result)** | The **weakest** leave-one-country-out result: its coefficient and its wild-bootstrap p-value (the worst case over 34 runs). |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. Here *robust share* is the share of all leave-one-out and alternative-definition runs with a negative coefficient and p < 0.10. |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf
import patsy

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · Leave one country out
The H0 model is re-estimated 34 times, each time without one country (wild bootstrap, 499 draws each). Countries are sorted by how much their removal changes the coefficient.

In [2]:
full = wild_cluster("dev_z", fy, outcome="roa_pp", B=1999)
rows = []
for iso in dim.index:
    r = wild_cluster("dev_z", fy[fy.iso3 != iso], outcome="roa_pp", B=499)
    rows.append({"dropped": iso, "country": names[iso], "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "firm-years": r["n"], "change vs full": r["coef"] - full["coef"]})
loco = pd.DataFrame(rows).set_index("dropped").sort_values("coef")
fig = px.scatter(loco.reset_index(), x="coef", y="dropped", error_x=loco.ci_high - loco.coef, error_x_minus=loco.coef - loco.ci_low, hover_name="country",
                 title=f"Development → ROA (pp per +1 SD) when each country is dropped (full sample: {full['coef']:+.2f})")
fig.add_vline(x=0, line_color="grey"); fig.add_vline(x=full["coef"], line_dash="dot", line_color="#E45756"); fig.update_layout(height=780, yaxis_autorange="reversed"); fig.show()
print(f"Full sample: {full['coef']:+.2f} pp (wild p = {full['p_wild']:.3f}). Leave-one-out range: {loco.coef.min():+.2f} to {loco.coef.max():+.2f}; "
      f"negative in {int((loco.coef < 0).sum())} of 34 runs, p < 0.10 in {int((loco['p (wild)'] < 0.10).sum())}, p < 0.05 in {int((loco['p (wild)'] < 0.05).sum())}.")
loco.head(8).round(3)

Full sample: -1.12 pp (wild p = 0.001). Leave-one-out range: -1.22 to -0.93; negative in 34 of 34 runs, p < 0.10 in 34, p < 0.05 in 34.


,country,coef,ci_low,ci_high,p (wild),firm-years,change vs full
dropped,,,,,,,
MEX,Mexico,-1.221,-1.710,-0.732,0.002,9837,-0.097
USA,United States,-1.186,-1.648,-0.724,0.002,9822,-0.063
ZAF,South Africa,-1.185,-1.702,-0.668,0.004,9828,-0.062
BRA,Brazil,-1.179,-1.673,-0.684,0.002,9828,-0.055
JPN,Japan,-1.174,-1.648,-0.700,0.002,9821,-0.051
AUS,Australia,-1.173,-1.651,-0.694,0.002,9830,-0.049
SWE,Sweden,-1.157,-1.645,-0.668,0.002,9821,-0.033
ROU,Romania,-1.137,-1.617,-0.657,0.002,10010,-0.013


### Drop the most influential countries cumulatively
Remove countries one at a time in order of influence (the biggest weakening of the link first) and watch the coefficient.

In [3]:
order = loco.sort_values("coef", ascending=False).index.tolist()        # least negative coef first = removal that weakens the link most
rows = []
for k in range(0, 6):
    dropped = order[:k]
    r = wild_cluster("dev_z", fy[~fy.iso3.isin(dropped)], outcome="roa_pp", B=499)
    rows.append({"countries dropped": k, "who": ", ".join(dropped) or "-", "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "countries left": r["clusters"]})
cum = pd.DataFrame(rows).set_index("countries dropped")
fig = go.Figure(go.Scatter(x=cum.index, y=cum.coef, mode="lines+markers", error_y=dict(type="data", array=cum.ci_high - cum.coef, arrayminus=cum.coef - cum.ci_low)))
fig.add_hline(y=0, line_color="grey"); fig.update_layout(height=380, title="Development → ROA as the most influential countries are removed", xaxis_title="number of countries removed", yaxis_title="pp per +1 SD"); fig.show()
cum.round(3)

,who,coef,ci_low,ci_high,p (wild),countries left
countries dropped,,,,,,
0,-,-1.124,-1.599,-0.648,0.002,34
1,IND,-0.928,-1.276,-0.580,0.002,33
2,"IND, NLD",-0.845,-1.179,-0.511,0.006,32
3,"IND, NLD, DNK",-0.757,-1.091,-0.424,0.002,31
4,"IND, NLD, DNK, BEL",-0.679,-1.000,-0.359,0.006,30
5,"IND, NLD, DNK, BEL, SGP",-0.609,-0.950,-0.268,0.010,29


## 2 · Other definitions
Same test with other measures of development and of ROA. Several measures here describe the same thing, so agreement is expected; disagreement would be informative.

In [4]:
zz = ((profile - profile.mean()) / profile.std()).fillna(0)
U, S, Vt = np.linalg.svd(zz.values, full_matrices=False)
pc1 = pd.Series(U[:, 0] * S[0], index=zz.index); pc1 = pc1 * np.sign(np.corrcoef(pc1, country["dev_idx"])[0, 1]); pc1 = (pc1 - pc1.mean()) / pc1.std()
fy["pc1_all_features"] = fy.iso3.map(pc1)
fy["dev_static"] = fy.iso3.map(country["dev_idx"])
fy["op_margin_pp"] = fy.operating_margin_w * 100
lo, hi = fy.roa.quantile([.05, .95]); fy["roa_pp_w5"] = fy.roa.clip(lo, hi) * 100
specs = [("development index, t-1  [primary]", "dev_z", "roa_pp"), ("development index, 2019–2023 average", "dev_static", "roa_pp"),
         ("first principal component of all 35 features", "pc1_all_features", "roa_pp"), ("GDP per person only (t-1)", "gdp_per_capita_ppp_z", "roa_pp"),
         ("life expectancy only (t-1)", "life_expectancy_z", "roa_pp"), ("institutions index (t-1)", "inst_z", "roa_pp"),
         ("development, ROA winsorised at 5%/95%", "dev_z", "roa_pp_w5"), ("development, ROA minus sector median", "dev_z", "roa_sector_adj_pp"),
         ("development, operating margin instead of ROA", "dev_z", "op_margin_pp")]
rows = []
for lab, f, out in specs:
    ctrl = CONTROLS if out != "roa_sector_adj_pp" else "size_rank + C(fiscal_year)"
    r = wild_cluster(f, fy, outcome=out, B=999, controls=ctrl)
    rows.append({"definition": lab, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "n": r["n"]})
alt = pd.DataFrame(rows).set_index("definition")
fig = px.scatter(alt.reset_index(), x="coef", y="definition", error_x=alt.ci_high - alt.coef, error_x_minus=alt.coef - alt.ci_low, title="Alternative definitions (coefficient per +1 SD of the country measure)")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=450, yaxis_autorange="reversed"); fig.show()
alt

,coef,ci_low,ci_high,p (wild),n
definition,,,,,
"development index, t-1 [primary]",-1.124,-1.599,-0.648,0.001,10220
"development index, 2019–2023 average",-1.094,-1.553,-0.636,0.001,10220
first principal component of all 35 features,-1.132,-1.566,-0.699,0.001,10220
GDP per person only (t-1),-1.115,-1.648,-0.581,0.006,10220
life expectancy only (t-1),-0.989,-1.556,-0.422,0.002,10220
institutions index (t-1),-1.016,-1.497,-0.536,0.001,10220
"development, ROA winsorised at 5%/95%",-0.744,-1.093,-0.394,0.001,10220
"development, ROA minus sector median",-1.137,-1.589,-0.684,0.001,10220
"development, operating margin instead of ROA",-3.512,-5.339,-1.686,0.006,9131


## 3 · Countries counted equally
In the firm-level model a country with 500 firms weighs more than one with 20. Here every country is one observation: the median ROA of its companies against its development index (Spearman, with a permutation p-value that shuffles the countries' development scores).

In [5]:
cm = fy.groupby("iso3").agg(roa=("roa_pp", "median"), n=("company_id", "nunique")).query("n >= @MIN_FIRMS").join(country["dev_idx"].rename("dev"))
rho_c, p, n = spearman(cm.dev, cm.roa)
rng = np.random.default_rng(0)
perm = np.array([stats.spearmanr(rng.permutation(cm.dev.values), cm.roa.values)[0] for _ in range(5000)])
p_perm = (np.sum(np.abs(perm) >= abs(rho_c)) + 1) / (len(perm) + 1)
lo, hi = boot_ci(cm.dev, cm.roa)
print(f"{n} countries: ρ = {rho_c:+.2f} (bootstrap 95% CI {lo:+.2f} to {hi:+.2f}); analytic p = {p:.4f}, permutation p = {p_perm:.4f}")
px.scatter(cm.reset_index(), x="dev", y="roa", text="iso3", title=f"Median ROA vs development, countries counted equally (ρ = {rho_c:+.2f})", labels={"dev": "development index", "roa": "median ROA (pp)"}, height=430).update_traces(textposition="top center")

34 countries: ρ = -0.35 (bootstrap 95% CI -0.60 to -0.03); analytic p = 0.0422, permutation p = 0.0408


## 4 · Placebo: shuffle the countries
The firm-level estimate is repeated 5,000 times with the countries' development scores randomly reassigned. If the real coefficient is far in the tail of this placebo distribution, the link is not a quirk of which country happens to be which. (Time-invariant development score, residualised on sector, size and year for speed.)

In [6]:
d = fy.dropna(subset=["roa_pp", "dev_static", "size_rank"])
y, X = patsy.dmatrices("roa_pp ~ C(sector) + size_rank + C(fiscal_year)", d, return_type="dataframe")
res = y.values[:, 0] - X.values @ np.linalg.lstsq(X.values, y.values[:, 0], rcond=None)[0]
code_ = pd.factorize(d.iso3)[0]; isos = d.iso3.unique()
score = d.groupby("iso3").dev_static.first().reindex(pd.unique(d.iso3)).values
def slope(sc):
    x = sc[code_]; x = x - x.mean(); return (x @ res) / (x @ x)
real = slope(score)
placebo = np.array([slope(rng.permutation(score)) for _ in range(5000)])
p_placebo = (np.sum(np.abs(placebo) >= abs(real)) + 1) / (len(placebo) + 1)
fig = px.histogram(x=placebo, nbins=60, title=f"Placebo coefficients (countries' scores shuffled) vs real {real:+.2f} (permutation p = {p_placebo:.4f})", labels={"x": "pp ROA per +1 SD"})
fig.add_vline(x=real, line_color="#E45756", annotation_text="real"); fig.update_layout(height=380, showlegend=False); fig.show()
print(f"Real coefficient (static score) = {real:+.2f}; placebo SD = {placebo.std():.2f}; permutation p = {p_placebo:.4f}")

Real coefficient (static score) = -1.09; placebo SD = 0.37; permutation p = 0.0012


## 5 · Replication on ESEF
The leave-one-country-out exercise on the independent EU source (about 19 countries, no sector control).

In [7]:
fe = prep.load_firms(con, country, source="ESEF")
fe_full = wild_cluster("dev_z", fe, outcome="roa_pp", controls="size_rank + C(fiscal_year)", B=1999)
rows = []
for iso in fe.iso3.unique():
    r = wild_cluster("dev_z", fe[fe.iso3 != iso], outcome="roa_pp", controls="size_rank + C(fiscal_year)", B=499)
    rows.append({"dropped": iso, "coef": r["coef"], "p (wild)": r["p_wild"]})
loco_e = pd.DataFrame(rows).set_index("dropped")
print(f"ESEF full: {fe_full['coef']:+.2f} (wild p = {fe_full['p_wild']:.3f}, {fe_full['clusters']} countries). Leave-one-out: {loco_e.coef.min():+.2f} to {loco_e.coef.max():+.2f}; negative in {int((loco_e.coef < 0).sum())} of {len(loco_e)}, p < 0.10 in {int((loco_e['p (wild)'] < 0.10).sum())}.")

ESEF full: -1.29 (wild p = 0.023, 19 countries). Leave-one-out: -1.56 to -1.08; negative in 19 of 19, p < 0.10 in 19.


## 6 · Verdict

In [8]:
worst = loco.sort_values("coef", ascending=False).iloc[0]
all_runs = pd.concat([loco[["coef", "p (wild)"]], alt[["coef", "p (wild)"]]])
robust_share = float(((all_runs.coef < 0) & (all_runs["p (wild)"] < 0.10)).mean())
v = utils.verdict(worst.coef, worst["p (wild)"], -1, robust_share, 33)
worst_e = loco_e.sort_values("coef", ascending=False).iloc[0]
rep_same = bool((loco_e.coef < 0).all())
lines = [
 f"Full sample (H0): {full['coef']:+.2f} pp. Weakest leave-one-country-out result: dropping {worst.country} gives {worst.coef:+.2f} pp (95% CI {worst.ci_low:+.2f} to {worst.ci_high:+.2f}), wild p = {worst['p (wild)']:.3f}.",
 f"Leave-one-out: negative in {int((loco.coef < 0).sum())} of 34 runs; p < 0.10 in {int((loco['p (wild)'] < 0.10).sum())}, p < 0.05 in {int((loco['p (wild)'] < 0.05).sum())}. Dropping the 5 most influential countries: {cum.coef.iloc[-1]:+.2f} pp (p = {cum['p (wild)'].iloc[-1]:.3f}).",
 f"Other definitions: negative and p < 0.10 in {int(((alt.coef < 0) & (alt['p (wild)'] < 0.10)).sum())} of {len(alt)}. Countries counted equally: ρ = {rho_c:+.2f} (permutation p = {p_perm:.3f}); placebo permutation p = {p_placebo:.3f}.",
 f"Share of all runs with negative coefficient and p < 0.10: {robust_share:.0%}. ESEF leave-one-out: negative in {int((loco_e.coef < 0).sum())} of {len(loco_e)} runs (weakest {worst_e.coef:+.2f}, p = {worst_e['p (wild)']:.3f}).",
 f"VERDICT H11: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H11", id="H11", title="Sensitivity of the development–ROA link", statement="The H0 link survives dropping any single country and alternative definitions.",
                  expected_sign=-1, effect=float(worst.coef), ci_low=float(worst.ci_low), ci_high=float(worst.ci_high), effect_unit="pp ROA per +1 SD development, weakest leave-one-country-out run",
                  p_primary=float(worst["p (wild)"]), p_method="wild cluster bootstrap (worst case of 34 runs)", n=int(worst["firm-years"]), clusters=33, robust_share=robust_share,
                  replication_coef=float(worst_e.coef), replication_p=float(worst_e["p (wild)"]), replication_same_sign=rep_same, verdict=v,
                  notebook="1.11_sensitivity_of_the_main_finding.ipynb", notes=lines)

Full sample (H0): -1.12 pp. Weakest leave-one-country-out result: dropping India gives -0.93 pp (95% CI -1.28 to -0.58), wild p = 0.002.
Leave-one-out: negative in 34 of 34 runs; p < 0.10 in 34, p < 0.05 in 34. Dropping the 5 most influential countries: -0.61 pp (p = 0.010).
Other definitions: negative and p < 0.10 in 9 of 9. Countries counted equally: ρ = -0.35 (permutation p = 0.041); placebo permutation p = 0.001.
Share of all runs with negative coefficient and p < 0.10: 100%. ESEF leave-one-out: negative in 19 of 19 runs (weakest -1.08, p = 0.038).
VERDICT H11: SUPPORTED


### Limits
- Leave-one-out tests influence of single countries, not of **groups** of countries that share a feature (for example all emerging markets); the cumulative removal partly covers this.
- All specifications use the same companies and the same sources: they test sensitivity to *choices*, not independence.
- Passing every check still does not turn an association into a cause.

In [9]:
con.close()